In [2]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import re
from dataclasses import dataclass
from threading import Lock

import serial
import serial.tools.list_ports

from PyQt6.QtCore import QThread, pyqtSignal
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox, QTableWidget, QTableWidgetItem,
    QTabWidget, QCheckBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# ============================================================
# COM PORT HELPERS
# ============================================================
def normalize_windows_com(port: str) -> str:
    port = port.strip().upper()
    if re.fullmatch(r"COM\d+", port) and int(port[3:]) >= 10:
        return r"\\.\{}".format(port)
    return port


def extract_display_com(port_info) -> str:
    text = f"{port_info.device} {port_info.description} {port_info.hwid}"
    matches = re.findall(r"COM\d+", text.upper())
    return matches[-1] if matches else port_info.device


def get_real_com_ports():
    ports = list(serial.tools.list_ports.comports())
    valid = []
    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()
        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "SERIAL" in desc
            or "UART" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )
        is_bluetooth = "BLUETOOTH" in desc or "BTHENUM" in hwid
        if is_usb_serial and not is_bluetooth:
            valid.append(p)
    return valid


# ============================================================
# SINGLE-PUMP SERIAL DRIVER
# ============================================================
class BT100Link:
    """One serial link per pump/USB adapter."""

    def __init__(self, port: str, addr: int):
        self.port = port
        self.addr = int(addr)
        self.ser = None
        self.lock = Lock()

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    @staticmethod
    def checksum(addr: int, pdu: bytes) -> int:
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    @staticmethod
    def escape(data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def frame(self, pdu: bytes) -> bytes:
        fcs = self.checksum(self.addr, pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self.escape(body)

    def send(self, pdu: bytes, wait_s=0.25):
        with self.lock:
            self.open()
            tx = self.frame(pdu)
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()
            self.ser.write(tx)
            self.ser.flush()
            time.sleep(wait_s)
            rx = self.ser.read_all()
            return tx, rx

    def set_speed(self, rpm: float, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        raw = int(round(rpm * 10))
        state1 = 0x01
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + raw.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx: bytes):
    if not rx or len(rx) < 10 or rx[0] != 0xE9:
        return None
    length = rx[2]
    pdu = rx[3:3 + length]
    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None
    raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]
    return {
        "rpm": raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


# ============================================================
# WORKERS
# ============================================================
class MonitorWorker(QThread):
    data = pyqtSignal(float, float, bool, str, str)
    warning = pyqtSignal(str)
    disconnected = pyqtSignal(str)

    def __init__(self, link: BT100Link, interval_s=0.5):
        super().__init__()
        self.link = link
        self.interval_s = interval_s
        self.running = True
        self.t0 = time.time()

    def stop(self):
        self.running = False

    def run(self):
        while self.running:
            try:
                _, rx = self.link.read_speed()
                decoded = decode_rpm(rx)
                if decoded is None:
                    self.warning.emit("No valid RPM response")
                else:
                    self.data.emit(
                        time.time() - self.t0,
                        decoded["rpm"],
                        decoded["running"],
                        decoded["direction"],
                        decoded["raw"],
                    )
            except serial.SerialException as e:
                self.disconnected.emit(str(e))
                break
            except Exception as e:
                self.warning.emit(str(e))
            time.sleep(self.interval_s)


@dataclass
class SequenceStep:
    rpm: float
    duration_s: float
    direction: str


class SequenceWorker(QThread):
    log = pyqtSignal(str)
    finished_ok = pyqtSignal()
    error = pyqtSignal(str)

    def __init__(self, link: BT100Link, steps):
        super().__init__()
        self.link = link
        self.steps = steps
        self.running = True

    def stop(self):
        self.running = False

    def run(self):
        try:
            for i, step in enumerate(self.steps, 1):
                if not self.running:
                    break
                cw = step.direction == "CW"
                self.log.emit(f"Step {i}: {step.rpm} RPM, {step.duration_s}s, {step.direction}")
                self.link.set_speed(step.rpm, cw=cw)
                t0 = time.time()
                while time.time() - t0 < step.duration_s:
                    if not self.running:
                        break
                    time.sleep(0.1)
                self.link.stop(cw=cw)
            self.link.stop()
            self.finished_ok.emit()
        except Exception as e:
            try:
                self.link.stop()
            except Exception:
                pass
            self.error.emit(str(e))


# ============================================================
# PLOT
# ============================================================
class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(3.6, 2.2), tight_layout=True)
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)
        self.title = title
        self.t = []
        self.actual = []
        self.target = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("s")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def add_point(self, t, actual_rpm, target_rpm):
        self.t.append(t)
        self.actual.append(actual_rpm)
        self.target.append(target_rpm)
        if len(self.t) > 240:
            self.t = self.t[-240:]
            self.actual = self.actual[-240:]
            self.target = self.target[-240:]
        self.ax.clear()
        self.ax.plot(self.t, self.actual, label="Actual")
        self.ax.plot(self.t, self.target, linestyle="--", label="Target")
        self.ax.set_title(self.title)
        self.ax.set_xlabel("s")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.ax.legend(loc="upper right", fontsize=8)
        self.draw()

    def clear(self):
        self.t.clear()
        self.actual.clear()
        self.target.clear()
        self.redraw()


# ============================================================
# PUMP PANEL
# ============================================================
class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)
        self.link = None
        self.monitor_worker = None
        self.sequence_worker = None
        self.target_rpm = 0.0

        self.port_box = QComboBox()
        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")
        self.interval_edit = QLineEdit("0.5")
        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.status_dot = QLabel("●")
        self.status_dot.setStyleSheet("color: gray; font-size: 18px;")
        self.status = QLabel("Disconnected")
        self.feedback = QLabel("Target - | Actual - | Error -")
        self.feedback.setStyleSheet("font-weight: 600;")

        self.compact_log = QCheckBox("Show log")
        self.compact_log.setChecked(False)
        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMaximumHeight(90)
        self.log.setVisible(False)

        self.plot = PlotCanvas(title)
        self.tabs = QTabWidget()
        self.sequence_table = QTableWidget(0, 3)
        self.sequence_table.setHorizontalHeaderLabels(["RPM", "s", "Dir"])
        self.sequence_table.horizontalHeader().setStretchLastSection(True)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        main = QVBoxLayout()
        top = QGridLayout()
        top.setHorizontalSpacing(6)
        top.setVerticalSpacing(4)

        btn_refresh = QPushButton("↻")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read")
        btn_monitor = QPushButton("Monitor")
        btn_stop_monitor = QPushButton("Stop Mon")
        btn_toggle = QPushButton("Toggle Dir")

        top.addWidget(self.status_dot, 0, 0)
        top.addWidget(self.status, 0, 1, 1, 3)
        top.addWidget(QLabel("COM"), 1, 0)
        top.addWidget(self.port_box, 1, 1, 1, 2)
        top.addWidget(btn_refresh, 1, 3)
        top.addWidget(QLabel("Addr"), 2, 0)
        top.addWidget(self.addr_edit, 2, 1)
        top.addWidget(QLabel("RPM"), 2, 2)
        top.addWidget(self.rpm_edit, 2, 3)
        top.addWidget(QLabel("Dir"), 3, 0)
        top.addWidget(self.dir_box, 3, 1)
        top.addWidget(QLabel("Poll"), 3, 2)
        top.addWidget(self.interval_edit, 3, 3)
        top.addWidget(btn_connect, 4, 0)
        top.addWidget(btn_disconnect, 4, 1)
        top.addWidget(btn_start, 4, 2)
        top.addWidget(btn_stop, 4, 3)
        top.addWidget(btn_read, 5, 0)
        top.addWidget(btn_monitor, 5, 1)
        top.addWidget(btn_stop_monitor, 5, 2)
        top.addWidget(btn_toggle, 5, 3)
        top.addWidget(self.feedback, 6, 0, 1, 4)

        plot_tab = QWidget()
        plot_lay = QVBoxLayout()
        plot_lay.setContentsMargins(2, 2, 2, 2)
        plot_lay.addWidget(self.plot)
        btn_clear = QPushButton("Clear Plot")
        btn_clear.clicked.connect(self.plot.clear)
        plot_lay.addWidget(btn_clear)
        plot_tab.setLayout(plot_lay)

        seq_tab = QWidget()
        seq_lay = QVBoxLayout()
        seq_lay.setContentsMargins(2, 2, 2, 2)
        seq_lay.addWidget(self.sequence_table)
        seq_buttons = QHBoxLayout()
        btn_add = QPushButton("Add")
        btn_remove = QPushButton("Remove")
        btn_run = QPushButton("Run")
        btn_stop_seq = QPushButton("Stop")
        seq_buttons.addWidget(btn_add)
        seq_buttons.addWidget(btn_remove)
        seq_buttons.addWidget(btn_run)
        seq_buttons.addWidget(btn_stop_seq)
        seq_lay.addLayout(seq_buttons)
        seq_tab.setLayout(seq_lay)

        self.tabs.addTab(plot_tab, "Plot")
        self.tabs.addTab(seq_tab, "Seq")

        main.addLayout(top)
        main.addWidget(self.tabs)
        main.addWidget(self.compact_log)
        main.addWidget(self.log)
        self.setLayout(main)

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_toggle.clicked.connect(self.toggle_direction)
        btn_add.clicked.connect(self.add_sequence_step)
        btn_remove.clicked.connect(self.remove_sequence_step)
        btn_run.clicked.connect(self.run_sequence)
        btn_stop_seq.clicked.connect(self.stop_sequence)
        self.compact_log.toggled.connect(self.log.setVisible)

    def set_status(self, text, color):
        self.status.setText(text)
        self.status_dot.setStyleSheet(f"color: {color}; font-size: 18px;")

    def refresh_ports(self):
        current = self.get_port(safe=True)
        self.port_box.clear()
        ports = get_real_com_ports()
        for p in ports:
            real = extract_display_com(p)
            open_port = normalize_windows_com(real)
            display = f"{real} | {p.description}"
            self.port_box.addItem(display, open_port)
        if ports:
            open_ports = [self.port_box.itemData(i) for i in range(self.port_box.count())]
            if current in open_ports:
                self.port_box.setCurrentIndex(open_ports.index(current))
        self.log.append("Ports refreshed")

    def get_port(self, safe=False):
        port = self.port_box.currentData()
        if not port and not safe:
            raise ValueError("No COM port selected")
        return port or ""

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be 1 to 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be 0 to 100")
        return rpm

    def get_interval(self):
        return max(0.2, float(self.interval_edit.text().strip()))

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_link(self):
        if self.link is None:
            self.connect_pump()
        if self.link is None:
            raise RuntimeError("Pump is not connected")
        self.link.addr = self.get_addr()
        return self.link

    def connect_pump(self):
        try:
            self.disconnect_pump(show_msg=False)
            self.link = BT100Link(self.get_port(), self.get_addr())
            self.link.open()
            self.set_status(f"Connected {self.get_port()}", "green")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")
        except Exception as e:
            self.link = None
            self.set_status("Connect failed", "red")
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_msg=True):
        self.stop_monitor()
        self.stop_sequence()
        try:
            if self.link:
                try:
                    self.link.stop(cw=self.get_cw())
                except Exception:
                    pass
                self.link.close()
            self.link = None
            self.set_status("Disconnected", "gray")
            if show_msg:
                self.log.append("Disconnected")
        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, tx, rx):
        self.log.append(f"TX: {tx.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}\n")

    def start_pump(self):
        try:
            self.target_rpm = self.get_rpm()
            tx, rx = self.ensure_link().set_speed(self.target_rpm, cw=self.get_cw())
            self.log_io(tx, rx)
            self.set_status(f"Running {self.target_rpm:.1f} {self.dir_box.currentText()}", "green")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            tx, rx = self.ensure_link().stop(cw=self.get_cw())
            self.target_rpm = 0.0
            self.log_io(tx, rx)
            self.set_status("Stopped", "orange")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            tx, rx = self.ensure_link().read_speed()
            self.log_io(tx, rx)
            data = decode_rpm(rx)
            if data:
                self.update_feedback(data["rpm"], data["running"], data["direction"])
            else:
                self.feedback.setText("No valid RPM response")
                self.set_status("No response", "orange")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Read Error", str(e))

    def update_feedback(self, actual, running, direction):
        error = actual - self.target_rpm
        self.feedback.setText(
            f"Target {self.target_rpm:.1f} | Actual {actual:.1f} | Error {error:+.1f} | {direction}"
        )
        self.set_status("Running" if running else "Stopped", "green" if running else "orange")

    def start_monitor(self):
        try:
            self.stop_monitor()
            worker = MonitorWorker(self.ensure_link(), interval_s=self.get_interval())
            worker.data.connect(self.handle_monitor_data)
            worker.warning.connect(self.handle_monitor_warning)
            worker.disconnected.connect(self.handle_disconnected)
            self.monitor_worker = worker
            worker.start()
            self.set_status("Monitoring", "green")
        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.monitor_worker and self.monitor_worker.isRunning():
            self.monitor_worker.stop()
            self.monitor_worker.wait(1500)
        self.monitor_worker = None

    def handle_monitor_data(self, t, actual, running, direction, raw):
        self.plot.add_point(t, actual, self.target_rpm)
        self.update_feedback(actual, running, direction)

    def handle_monitor_warning(self, msg):
        self.feedback.setText(msg)
        self.set_status("Warn", "orange")

    def handle_disconnected(self, msg):
        self.set_status("Disconnected", "red")
        self.feedback.setText("COM disconnected")
        try:
            if self.link:
                self.link.close()
        except Exception:
            pass
        self.link = None

    def toggle_direction(self):
        self.dir_box.setCurrentText("CCW" if self.dir_box.currentText() == "CW" else "CW")

    def add_sequence_step(self):
        row = self.sequence_table.rowCount()
        self.sequence_table.insertRow(row)
        values = [self.rpm_edit.text().strip(), "5", self.dir_box.currentText()]
        for col, value in enumerate(values):
            self.sequence_table.setItem(row, col, QTableWidgetItem(value))

    def remove_sequence_step(self):
        row = self.sequence_table.currentRow()
        if row >= 0:
            self.sequence_table.removeRow(row)

    def get_steps(self):
        steps = []
        for row in range(self.sequence_table.rowCount()):
            rpm_item = self.sequence_table.item(row, 0)
            dur_item = self.sequence_table.item(row, 1)
            dir_item = self.sequence_table.item(row, 2)
            if not all([rpm_item, dur_item, dir_item]):
                raise ValueError(f"Missing value in row {row + 1}")
            direction = dir_item.text().strip().upper()
            if direction not in ["CW", "CCW"]:
                raise ValueError(f"Direction must be CW or CCW in row {row + 1}")
            steps.append(SequenceStep(float(rpm_item.text()), float(dur_item.text()), direction))
        return steps

    def run_sequence(self):
        try:
            if self.sequence_worker and self.sequence_worker.isRunning():
                QMessageBox.warning(self, "Sequence", "Sequence already running")
                return
            steps = self.get_steps()
            if not steps:
                QMessageBox.information(self, "Sequence", "No steps added")
                return
            worker = SequenceWorker(self.ensure_link(), steps)
            worker.log.connect(self.log.append)
            worker.finished_ok.connect(lambda: self.set_status("Seq done", "orange"))
            worker.error.connect(lambda e: QMessageBox.critical(self, "Sequence Error", e))
            self.sequence_worker = worker
            worker.start()
            self.set_status("Seq running", "green")
        except Exception as e:
            QMessageBox.critical(self, "Sequence Error", str(e))

    def stop_sequence(self):
        if self.sequence_worker and self.sequence_worker.isRunning():
            self.sequence_worker.stop()
            self.sequence_worker.wait(1500)
        self.sequence_worker = None

    def emergency_stop(self):
        try:
            self.stop_monitor()
            self.stop_sequence()
            if self.link:
                self.link.stop(cw=self.get_cw())
            self.target_rpm = 0.0
            self.set_status("E-Stop", "red")
        except Exception:
            pass


# ============================================================
# MAIN WINDOW
# ============================================================
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("BT100-1L Dual Pump Controller")
        self.resize(1200, 720)

        self.pump1 = PumpPanel("Pump 1", 1)
        self.pump2 = PumpPanel("Pump 2", 1)

        layout = QVBoxLayout()
        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        controls = QHBoxLayout()
        btn_refresh = QPushButton("Refresh Ports")
        btn_start = QPushButton("Start Both")
        btn_stop = QPushButton("Stop Both")
        btn_monitor = QPushButton("Monitor Both")
        btn_stop_monitor = QPushButton("Stop Monitors")
        btn_estop = QPushButton("EMERGENCY STOP")
        btn_estop.setStyleSheet("background-color: #b00020; color: white; font-weight: bold;")

        btn_refresh.clicked.connect(lambda: [self.pump1.refresh_ports(), self.pump2.refresh_ports()])
        btn_start.clicked.connect(lambda: [self.pump1.start_pump(), self.pump2.start_pump()])
        btn_stop.clicked.connect(lambda: [self.pump1.stop_pump(), self.pump2.stop_pump()])
        btn_monitor.clicked.connect(lambda: [self.pump1.start_monitor(), self.pump2.start_monitor()])
        btn_stop_monitor.clicked.connect(lambda: [self.pump1.stop_monitor(), self.pump2.stop_monitor()])
        btn_estop.clicked.connect(self.emergency_stop)

        for btn in [btn_refresh, btn_start, btn_stop, btn_monitor, btn_stop_monitor, btn_estop]:
            controls.addWidget(btn)
        controls.addStretch()

        note = QLabel(
            "Each pump uses its own USB/COM port. Select the correct COM for each pump, then connect them independently."
        )
        note.setStyleSheet("color: #555;")
        note.setWordWrap(True)

        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)
        self.setLayout(layout)

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        self.pump1.disconnect_pump(show_msg=False)
        self.pump2.disconnect_pump(show_msg=False)
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyleSheet("""
        QWidget { font-size: 11px; }
        QGroupBox { font-weight: bold; border: 1px solid #cfcfcf; border-radius: 7px; margin-top: 8px; padding: 6px; }
        QGroupBox::title { subcontrol-origin: margin; left: 10px; padding: 0 3px; }
        QPushButton { padding: 4px 8px; border-radius: 4px; }
        QLineEdit, QComboBox { padding: 2px; min-height: 20px; }
        QTabWidget::pane { border: 1px solid #ddd; }
    """)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())


SystemExit: 0

In [1]:
# pip install pyqt6 pyserial matplotlib

import sys
import time
import re
from dataclasses import dataclass
from threading import Lock

import serial
import serial.tools.list_ports

from PyQt6.QtCore import QThread, pyqtSignal
from PyQt6.QtWidgets import (
    QApplication, QWidget, QLabel, QPushButton, QComboBox,
    QLineEdit, QTextEdit, QGridLayout, QHBoxLayout, QVBoxLayout,
    QGroupBox, QMessageBox, QTableWidget, QTableWidgetItem,
    QTabWidget, QCheckBox
)

from matplotlib.backends.backend_qtagg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


# ============================================================
# COM PORT HELPERS
# ============================================================
def normalize_windows_com(port: str) -> str:
    port = port.strip().upper()
    if re.fullmatch(r"COM\d+", port) and int(port[3:]) >= 10:
        return r"\\.\{}".format(port)
    return port


def extract_display_com(port_info) -> str:
    text = f"{port_info.device} {port_info.description} {port_info.hwid}"
    matches = re.findall(r"COM\d+", text.upper())
    return matches[-1] if matches else port_info.device


def get_real_com_ports():
    ports = list(serial.tools.list_ports.comports())
    valid = []
    for p in ports:
        desc = (p.description or "").upper()
        hwid = (p.hwid or "").upper()
        manufacturer = (p.manufacturer or "").upper()
        is_usb_serial = (
            p.vid is not None
            or "USB" in desc
            or "USB" in hwid
            or "SERIAL" in desc
            or "UART" in desc
            or "MOXA" in desc
            or "MOXA" in manufacturer
            or "UPORT" in desc
            or "UPORT" in manufacturer
        )
        is_bluetooth = "BLUETOOTH" in desc or "BTHENUM" in hwid
        if is_usb_serial and not is_bluetooth:
            valid.append(p)
    return valid


# ============================================================
# SINGLE-PUMP SERIAL DRIVER
# ============================================================
class BT100Link:
    """One serial link per pump/USB adapter."""

    def __init__(self, port: str, addr: int):
        self.port = port
        self.addr = int(addr)
        self.ser = None
        self.lock = Lock()

    def open(self):
        if self.ser is None or not self.ser.is_open:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=1200,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_EVEN,
                stopbits=serial.STOPBITS_ONE,
                timeout=1.0,
            )

    def close(self):
        if self.ser and self.ser.is_open:
            self.ser.close()
        self.ser = None

    @staticmethod
    def checksum(addr: int, pdu: bytes) -> int:
        x = addr ^ len(pdu)
        for b in pdu:
            x ^= b
        return x

    @staticmethod
    def escape(data: bytes) -> bytes:
        out = bytearray()
        for b in data:
            if b == 0xE8:
                out.extend([0xE8, 0x00])
            elif b == 0xE9:
                out.extend([0xE8, 0x01])
            else:
                out.append(b)
        return bytes(out)

    def frame(self, pdu: bytes) -> bytes:
        fcs = self.checksum(self.addr, pdu)
        body = bytes([self.addr, len(pdu)]) + pdu + bytes([fcs])
        return bytes([0xE9]) + self.escape(body)

    def send(self, pdu: bytes, wait_s=0.25):
        with self.lock:
            self.open()
            tx = self.frame(pdu)
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()
            self.ser.write(tx)
            self.ser.flush()
            time.sleep(wait_s)
            rx = self.ser.read_all()
            return tx, rx

    def set_speed(self, rpm: float, cw=True):
        rpm = max(0.0, min(float(rpm), 100.0))
        raw = int(round(rpm * 10))
        state1 = 0x01
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + raw.to_bytes(2, "big") + bytes([state1, state2])
        return self.send(pdu)

    def stop(self, cw=True):
        state1 = 0x00
        state2 = 0x01 if cw else 0x00
        pdu = b"XL" + b"\x00\x00" + bytes([state1, state2])
        return self.send(pdu)

    def read_speed(self):
        return self.send(b"DL", wait_s=0.45)


def decode_rpm(rx: bytes):
    if not rx or len(rx) < 10 or rx[0] != 0xE9:
        return None
    length = rx[2]
    pdu = rx[3:3 + length]
    if len(pdu) != 6 or pdu[:2] != b"DL":
        return None
    raw = int.from_bytes(pdu[2:4], "big")
    state1 = pdu[4]
    state2 = pdu[5]
    return {
        "rpm": raw / 10.0,
        "running": bool(state1 & 0x01),
        "direction": "CW" if state2 & 0x01 else "CCW",
        "raw": rx.hex(" "),
    }


# ============================================================
# WORKERS
# ============================================================
class MonitorWorker(QThread):
    data = pyqtSignal(float, float, bool, str, str)
    warning = pyqtSignal(str)
    disconnected = pyqtSignal(str)

    def __init__(self, link: BT100Link, interval_s=0.5):
        super().__init__()
        self.link = link
        self.interval_s = interval_s
        self.running = True
        self.t0 = time.time()

    def stop(self):
        self.running = False

    def run(self):
        while self.running:
            try:
                _, rx = self.link.read_speed()
                decoded = decode_rpm(rx)
                if decoded is None:
                    self.warning.emit("No valid RPM response")
                else:
                    self.data.emit(
                        time.time() - self.t0,
                        decoded["rpm"],
                        decoded["running"],
                        decoded["direction"],
                        decoded["raw"],
                    )
            except serial.SerialException as e:
                self.disconnected.emit(str(e))
                break
            except Exception as e:
                self.warning.emit(str(e))
            time.sleep(self.interval_s)


@dataclass
class SequenceStep:
    rpm: float
    duration_s: float
    direction: str


class SequenceWorker(QThread):
    log = pyqtSignal(str)
    finished_ok = pyqtSignal()
    error = pyqtSignal(str)

    def __init__(self, link: BT100Link, steps):
        super().__init__()
        self.link = link
        self.steps = steps
        self.running = True

    def stop(self):
        self.running = False

    def run(self):
        try:
            for i, step in enumerate(self.steps, 1):
                if not self.running:
                    break
                cw = step.direction == "CW"
                self.log.emit(f"Step {i}: {step.rpm} RPM, {step.duration_s}s, {step.direction}")
                self.link.set_speed(step.rpm, cw=cw)
                t0 = time.time()
                while time.time() - t0 < step.duration_s:
                    if not self.running:
                        break
                    time.sleep(0.1)
                self.link.stop(cw=cw)
            self.link.stop()
            self.finished_ok.emit()
        except Exception as e:
            try:
                self.link.stop()
            except Exception:
                pass
            self.error.emit(str(e))


# ============================================================
# PLOT
# ============================================================
class PlotCanvas(FigureCanvas):
    def __init__(self, title):
        self.fig = Figure(figsize=(3.6, 2.2), tight_layout=True)
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)
        self.title = title
        self.t = []
        self.actual = []
        self.target = []
        self.redraw()

    def redraw(self):
        self.ax.clear()
        self.ax.set_title(self.title)
        self.ax.set_xlabel("s")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.draw()

    def add_point(self, t, actual_rpm, target_rpm):
        self.t.append(t)
        self.actual.append(actual_rpm)
        self.target.append(target_rpm)
        if len(self.t) > 240:
            self.t = self.t[-240:]
            self.actual = self.actual[-240:]
            self.target = self.target[-240:]
        self.ax.clear()
        self.ax.plot(self.t, self.actual, label="Actual")
        self.ax.plot(self.t, self.target, linestyle="--", label="Target")
        self.ax.set_title(self.title)
        self.ax.set_xlabel("s")
        self.ax.set_ylabel("RPM")
        self.ax.grid(True)
        self.ax.legend(loc="upper right", fontsize=8)
        self.draw()

    def clear(self):
        self.t.clear()
        self.actual.clear()
        self.target.clear()
        self.redraw()


# ============================================================
# PUMP PANEL
# ============================================================
class PumpPanel(QGroupBox):
    def __init__(self, title, default_addr):
        super().__init__(title)
        self.link = None
        self.monitor_worker = None
        self.sequence_worker = None
        self.target_rpm = 0.0

        self.port_box = QComboBox()
        self.addr_edit = QLineEdit(str(default_addr))
        self.rpm_edit = QLineEdit("30.0")
        self.interval_edit = QLineEdit("0.5")
        self.dir_box = QComboBox()
        self.dir_box.addItems(["CW", "CCW"])

        self.status_dot = QLabel("●")
        self.status_dot.setStyleSheet("color: gray; font-size: 18px;")
        self.status = QLabel("Disconnected")
        self.feedback = QLabel("Target - | Actual - | Error -")
        self.feedback.setStyleSheet("font-weight: 600;")

        self.compact_log = QCheckBox("Show log")
        self.compact_log.setChecked(False)
        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setMaximumHeight(90)
        self.log.setVisible(False)

        self.plot = PlotCanvas(title)
        self.tabs = QTabWidget()
        self.sequence_table = QTableWidget(0, 3)
        self.sequence_table.setHorizontalHeaderLabels(["RPM", "s", "Dir"])
        self.sequence_table.horizontalHeader().setStretchLastSection(True)

        self.build_ui()
        self.refresh_ports()

    def build_ui(self):
        main = QVBoxLayout()
        top = QGridLayout()
        top.setHorizontalSpacing(6)
        top.setVerticalSpacing(4)

        btn_refresh = QPushButton("↻")
        btn_connect = QPushButton("Connect")
        btn_disconnect = QPushButton("Disconnect")
        btn_start = QPushButton("Start")
        btn_stop = QPushButton("Stop")
        btn_read = QPushButton("Read")
        btn_monitor = QPushButton("Monitor")
        btn_stop_monitor = QPushButton("Stop Mon")
        btn_toggle = QPushButton("Toggle Dir")
        btn_start.setObjectName("startButton")
        btn_stop.setObjectName("stopButton")
        btn_monitor.setObjectName("primaryButton")

        top.addWidget(self.status_dot, 0, 0)
        top.addWidget(self.status, 0, 1, 1, 3)
        top.addWidget(QLabel("COM"), 1, 0)
        top.addWidget(self.port_box, 1, 1, 1, 2)
        top.addWidget(btn_refresh, 1, 3)
        top.addWidget(QLabel("Addr"), 2, 0)
        top.addWidget(self.addr_edit, 2, 1)
        top.addWidget(QLabel("RPM"), 2, 2)
        top.addWidget(self.rpm_edit, 2, 3)
        top.addWidget(QLabel("Dir"), 3, 0)
        top.addWidget(self.dir_box, 3, 1)
        top.addWidget(QLabel("Poll"), 3, 2)
        top.addWidget(self.interval_edit, 3, 3)
        top.addWidget(btn_connect, 4, 0)
        top.addWidget(btn_disconnect, 4, 1)
        top.addWidget(btn_start, 4, 2)
        top.addWidget(btn_stop, 4, 3)
        top.addWidget(btn_read, 5, 0)
        top.addWidget(btn_monitor, 5, 1)
        top.addWidget(btn_stop_monitor, 5, 2)
        top.addWidget(btn_toggle, 5, 3)
        top.addWidget(self.feedback, 6, 0, 1, 4)

        plot_tab = QWidget()
        plot_lay = QVBoxLayout()
        plot_lay.setContentsMargins(2, 2, 2, 2)
        plot_lay.addWidget(self.plot)
        btn_clear = QPushButton("Clear Plot")
        btn_clear.clicked.connect(self.plot.clear)
        plot_lay.addWidget(btn_clear)
        plot_tab.setLayout(plot_lay)

        seq_tab = QWidget()
        seq_lay = QVBoxLayout()
        seq_lay.setContentsMargins(2, 2, 2, 2)
        seq_lay.addWidget(self.sequence_table)
        seq_buttons = QHBoxLayout()
        btn_add = QPushButton("Add")
        btn_remove = QPushButton("Remove")
        btn_run = QPushButton("Run")
        btn_stop_seq = QPushButton("Stop")
        seq_buttons.addWidget(btn_add)
        seq_buttons.addWidget(btn_remove)
        seq_buttons.addWidget(btn_run)
        seq_buttons.addWidget(btn_stop_seq)
        seq_lay.addLayout(seq_buttons)
        seq_tab.setLayout(seq_lay)

        self.tabs.addTab(plot_tab, "Plot")
        self.tabs.addTab(seq_tab, "Seq")

        main.addLayout(top)
        main.addWidget(self.tabs)
        main.addWidget(self.compact_log)
        main.addWidget(self.log)
        self.setLayout(main)

        btn_refresh.clicked.connect(self.refresh_ports)
        btn_connect.clicked.connect(self.connect_pump)
        btn_disconnect.clicked.connect(self.disconnect_pump)
        btn_start.clicked.connect(self.start_pump)
        btn_stop.clicked.connect(self.stop_pump)
        btn_read.clicked.connect(self.read_rpm)
        btn_monitor.clicked.connect(self.start_monitor)
        btn_stop_monitor.clicked.connect(self.stop_monitor)
        btn_toggle.clicked.connect(self.toggle_direction)
        btn_add.clicked.connect(self.add_sequence_step)
        btn_remove.clicked.connect(self.remove_sequence_step)
        btn_run.clicked.connect(self.run_sequence)
        btn_stop_seq.clicked.connect(self.stop_sequence)
        self.compact_log.toggled.connect(self.log.setVisible)

    def set_status(self, text, color):
        self.status.setText(text)
        self.status_dot.setStyleSheet(f"color: {color}; font-size: 18px;")

    def refresh_ports(self):
        current = self.get_port(safe=True)
        self.port_box.clear()
        ports = get_real_com_ports()
        for p in ports:
            real = extract_display_com(p)
            open_port = normalize_windows_com(real)
            display = f"{real} | {p.description}"
            self.port_box.addItem(display, open_port)
        if ports:
            open_ports = [self.port_box.itemData(i) for i in range(self.port_box.count())]
            if current in open_ports:
                self.port_box.setCurrentIndex(open_ports.index(current))
        self.log.append("Ports refreshed")

    def get_port(self, safe=False):
        port = self.port_box.currentData()
        if not port and not safe:
            raise ValueError("No COM port selected")
        return port or ""

    def get_addr(self):
        addr = int(self.addr_edit.text().strip())
        if not 1 <= addr <= 31:
            raise ValueError("Address must be 1 to 31")
        return addr

    def get_rpm(self):
        rpm = float(self.rpm_edit.text().strip())
        if not 0 <= rpm <= 100:
            raise ValueError("RPM must be 0 to 100")
        return rpm

    def get_interval(self):
        return max(0.2, float(self.interval_edit.text().strip()))

    def get_cw(self):
        return self.dir_box.currentText() == "CW"

    def ensure_link(self):
        if self.link is None:
            self.connect_pump()
        if self.link is None:
            raise RuntimeError("Pump is not connected")
        self.link.addr = self.get_addr()
        return self.link

    def connect_pump(self):
        try:
            self.disconnect_pump(show_msg=False)
            self.link = BT100Link(self.get_port(), self.get_addr())
            self.link.open()
            self.set_status(f"Connected {self.get_port()}", "green")
            self.log.append(f"Connected {self.get_port()} addr={self.get_addr()}")
        except Exception as e:
            self.link = None
            self.set_status("Connect failed", "red")
            QMessageBox.critical(self, "Connection Error", str(e))

    def disconnect_pump(self, show_msg=True):
        self.stop_monitor()
        self.stop_sequence()
        try:
            if self.link:
                try:
                    self.link.stop(cw=self.get_cw())
                except Exception:
                    pass
                self.link.close()
            self.link = None
            self.set_status("Disconnected", "gray")
            if show_msg:
                self.log.append("Disconnected")
        except Exception as e:
            QMessageBox.critical(self, "Disconnect Error", str(e))

    def log_io(self, tx, rx):
        self.log.append(f"TX: {tx.hex(' ')}")
        self.log.append(f"RX: {rx.hex(' ') if rx else '<no response>'}\n")

    def start_pump(self):
        try:
            self.target_rpm = self.get_rpm()
            tx, rx = self.ensure_link().set_speed(self.target_rpm, cw=self.get_cw())
            self.log_io(tx, rx)
            self.set_status(f"Running {self.target_rpm:.1f} {self.dir_box.currentText()}", "green")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Start Error", str(e))

    def stop_pump(self):
        try:
            tx, rx = self.ensure_link().stop(cw=self.get_cw())
            self.target_rpm = 0.0
            self.log_io(tx, rx)
            self.set_status("Stopped", "orange")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Stop Error", str(e))

    def read_rpm(self):
        try:
            tx, rx = self.ensure_link().read_speed()
            self.log_io(tx, rx)
            data = decode_rpm(rx)
            if data:
                self.update_feedback(data["rpm"], data["running"], data["direction"])
            else:
                self.feedback.setText("No valid RPM response")
                self.set_status("No response", "orange")
        except Exception as e:
            self.set_status("Error", "red")
            QMessageBox.critical(self, "Read Error", str(e))

    def update_feedback(self, actual, running, direction):
        error = actual - self.target_rpm
        self.feedback.setText(
            f"Target {self.target_rpm:.1f} | Actual {actual:.1f} | Error {error:+.1f} | {direction}"
        )
        self.set_status("Running" if running else "Stopped", "green" if running else "orange")

    def start_monitor(self):
        try:
            self.stop_monitor()
            worker = MonitorWorker(self.ensure_link(), interval_s=self.get_interval())
            worker.data.connect(self.handle_monitor_data)
            worker.warning.connect(self.handle_monitor_warning)
            worker.disconnected.connect(self.handle_disconnected)
            self.monitor_worker = worker
            worker.start()
            self.set_status("Monitoring", "green")
        except Exception as e:
            QMessageBox.critical(self, "Monitor Error", str(e))

    def stop_monitor(self):
        if self.monitor_worker and self.monitor_worker.isRunning():
            self.monitor_worker.stop()
            self.monitor_worker.wait(1500)
        self.monitor_worker = None

    def handle_monitor_data(self, t, actual, running, direction, raw):
        self.plot.add_point(t, actual, self.target_rpm)
        self.update_feedback(actual, running, direction)

    def handle_monitor_warning(self, msg):
        self.feedback.setText(msg)
        self.set_status("Warn", "orange")

    def handle_disconnected(self, msg):
        self.set_status("Disconnected", "red")
        self.feedback.setText("COM disconnected")
        try:
            if self.link:
                self.link.close()
        except Exception:
            pass
        self.link = None

    def toggle_direction(self):
        self.dir_box.setCurrentText("CCW" if self.dir_box.currentText() == "CW" else "CW")

    def add_sequence_step(self):
        row = self.sequence_table.rowCount()
        self.sequence_table.insertRow(row)
        values = [self.rpm_edit.text().strip(), "5", self.dir_box.currentText()]
        for col, value in enumerate(values):
            self.sequence_table.setItem(row, col, QTableWidgetItem(value))

    def remove_sequence_step(self):
        row = self.sequence_table.currentRow()
        if row >= 0:
            self.sequence_table.removeRow(row)

    def get_steps(self):
        steps = []
        for row in range(self.sequence_table.rowCount()):
            rpm_item = self.sequence_table.item(row, 0)
            dur_item = self.sequence_table.item(row, 1)
            dir_item = self.sequence_table.item(row, 2)
            if not all([rpm_item, dur_item, dir_item]):
                raise ValueError(f"Missing value in row {row + 1}")
            direction = dir_item.text().strip().upper()
            if direction not in ["CW", "CCW"]:
                raise ValueError(f"Direction must be CW or CCW in row {row + 1}")
            steps.append(SequenceStep(float(rpm_item.text()), float(dur_item.text()), direction))
        return steps

    def run_sequence(self):
        try:
            if self.sequence_worker and self.sequence_worker.isRunning():
                QMessageBox.warning(self, "Sequence", "Sequence already running")
                return
            steps = self.get_steps()
            if not steps:
                QMessageBox.information(self, "Sequence", "No steps added")
                return
            worker = SequenceWorker(self.ensure_link(), steps)
            worker.log.connect(self.log.append)
            worker.finished_ok.connect(lambda: self.set_status("Seq done", "orange"))
            worker.error.connect(lambda e: QMessageBox.critical(self, "Sequence Error", e))
            self.sequence_worker = worker
            worker.start()
            self.set_status("Seq running", "green")
        except Exception as e:
            QMessageBox.critical(self, "Sequence Error", str(e))

    def stop_sequence(self):
        if self.sequence_worker and self.sequence_worker.isRunning():
            self.sequence_worker.stop()
            self.sequence_worker.wait(1500)
        self.sequence_worker = None

    def emergency_stop(self):
        try:
            self.stop_monitor()
            self.stop_sequence()
            if self.link:
                self.link.stop(cw=self.get_cw())
            self.target_rpm = 0.0
            self.set_status("E-Stop", "red")
        except Exception:
            pass


# ============================================================
# MAIN WINDOW
# ============================================================
class MainWindow(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("BT100-1L Dual Pump Controller")
        self.resize(1200, 720)

        self.pump1 = PumpPanel("Pump 1", 1)
        self.pump2 = PumpPanel("Pump 2", 1)

        layout = QVBoxLayout()
        panels = QHBoxLayout()
        panels.addWidget(self.pump1)
        panels.addWidget(self.pump2)

        controls = QHBoxLayout()
        btn_refresh = QPushButton("Refresh Ports")
        btn_start = QPushButton("Start Both")
        btn_stop = QPushButton("Stop Both")
        btn_monitor = QPushButton("Monitor Both")
        btn_stop_monitor = QPushButton("Stop Monitors")
        btn_estop = QPushButton("EMERGENCY STOP")
        btn_start.setObjectName("startButton")
        btn_stop.setObjectName("stopButton")
        btn_monitor.setObjectName("primaryButton")
        btn_estop.setObjectName("dangerButton")

        btn_refresh.clicked.connect(lambda: [self.pump1.refresh_ports(), self.pump2.refresh_ports()])
        btn_start.clicked.connect(lambda: [self.pump1.start_pump(), self.pump2.start_pump()])
        btn_stop.clicked.connect(lambda: [self.pump1.stop_pump(), self.pump2.stop_pump()])
        btn_monitor.clicked.connect(lambda: [self.pump1.start_monitor(), self.pump2.start_monitor()])
        btn_stop_monitor.clicked.connect(lambda: [self.pump1.stop_monitor(), self.pump2.stop_monitor()])
        btn_estop.clicked.connect(self.emergency_stop)

        for btn in [btn_refresh, btn_start, btn_stop, btn_monitor, btn_stop_monitor, btn_estop]:
            controls.addWidget(btn)
        controls.addStretch()

        note = QLabel(
            "Each pump uses its own USB/COM port. Select the correct COM for each pump, then connect them independently."
        )
        note.setStyleSheet("color: #555;")
        note.setWordWrap(True)

        layout.addLayout(panels)
        layout.addLayout(controls)
        layout.addWidget(note)
        self.setLayout(layout)

    def emergency_stop(self):
        self.pump1.emergency_stop()
        self.pump2.emergency_stop()
        QMessageBox.information(self, "Emergency Stop", "Stop commands sent.")

    def closeEvent(self, event):
        self.pump1.disconnect_pump(show_msg=False)
        self.pump2.disconnect_pump(show_msg=False)
        event.accept()


if __name__ == "__main__":
    app = QApplication(sys.argv)
    app.setStyle("Fusion")
    app.setStyleSheet("""
        QWidget {
            font-family: "Segoe UI", Arial, sans-serif;
            font-size: 12px;
            color: #172033;
            background: #f4f7fb;
        }

        QGroupBox {
            background: #ffffff;
            border: 1px solid #d7deea;
            border-radius: 12px;
            margin-top: 12px;
            padding: 10px;
            font-weight: 700;
        }

        QGroupBox::title {
            subcontrol-origin: margin;
            left: 14px;
            padding: 0 6px;
            color: #1f3a5f;
            background: #f4f7fb;
        }

        QLabel {
            background: transparent;
        }

        QLineEdit, QComboBox {
            background: #ffffff;
            border: 1px solid #b8c4d6;
            border-radius: 7px;
            padding: 5px 7px;
            min-height: 24px;
        }

        QLineEdit:focus, QComboBox:focus {
            border: 2px solid #2f80ed;
            padding: 4px 6px;
        }

        QComboBox::drop-down {
            border: none;
            width: 24px;
        }

        QPushButton {
            background: #e9eff8;
            color: #172033;
            border: 1px solid #b9c7da;
            border-radius: 8px;
            padding: 7px 12px;
            font-weight: 650;
            min-height: 28px;
        }

        QPushButton:hover {
            background: #dbe7f7;
            border: 1px solid #8ea9ce;
        }

        QPushButton:pressed {
            background: #c5d7ee;
            padding-top: 8px;
            padding-bottom: 6px;
        }

        QPushButton:disabled {
            background: #edf1f5;
            color: #9aa7b8;
            border-color: #d6dde6;
        }

        QPushButton#startButton {
            background: #1f9d55;
            color: white;
            border: 1px solid #168345;
        }
        QPushButton#startButton:hover { background: #26b866; }
        QPushButton#startButton:pressed { background: #167a41; }

        QPushButton#stopButton {
            background: #f2994a;
            color: white;
            border: 1px solid #d37b2d;
        }
        QPushButton#stopButton:hover { background: #f5a85f; }
        QPushButton#stopButton:pressed { background: #cf7426; }

        QPushButton#dangerButton {
            background: #d62828;
            color: white;
            border: 1px solid #a91f1f;
            font-weight: 800;
        }
        QPushButton#dangerButton:hover { background: #ef3b3b; }
        QPushButton#dangerButton:pressed { background: #a91f1f; }

        QPushButton#primaryButton {
            background: #2f80ed;
            color: white;
            border: 1px solid #1f6fd1;
        }
        QPushButton#primaryButton:hover { background: #4593f4; }
        QPushButton#primaryButton:pressed { background: #1f6fd1; }

        QTextEdit {
            background: #0f172a;
            color: #dbeafe;
            border: 1px solid #22314d;
            border-radius: 8px;
            padding: 6px;
            font-family: Consolas, "Courier New", monospace;
            font-size: 11px;
        }

        QTabWidget::pane {
            border: 1px solid #d7deea;
            border-radius: 8px;
            background: #ffffff;
            top: -1px;
        }

        QTabBar::tab {
            background: #e9eff8;
            border: 1px solid #d7deea;
            border-bottom: none;
            border-top-left-radius: 8px;
            border-top-right-radius: 8px;
            padding: 7px 16px;
            margin-right: 2px;
            font-weight: 650;
        }

        QTabBar::tab:selected {
            background: #ffffff;
            color: #2f80ed;
        }

        QTableWidget {
            background: #ffffff;
            border: 1px solid #d7deea;
            border-radius: 8px;
            gridline-color: #e6ebf3;
            selection-background-color: #d8e9ff;
        }

        QHeaderView::section {
            background: #edf3fb;
            color: #1f3a5f;
            border: none;
            border-right: 1px solid #d7deea;
            padding: 5px;
            font-weight: 700;
        }

        QCheckBox {
            spacing: 6px;
            background: transparent;
        }
    """)
    win = MainWindow()
    win.show()
    sys.exit(app.exec())


SystemExit: 0

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
